# CONFIGURATION

# Cấu hình Hyperparameters cho các Model

Dưới đây là các đề xuất cấu hình (hyperparameters) để fine-tune cho từng nhóm model trong danh sách.

## 1. Cấu hình Chung (Áp dụng cho tất cả model)

Đây là các giá trị khởi điểm an toàn và phổ biến:

- **`EPOCHS`**: `3` (Thường là đủ cho NLI, có thể tăng lên `4` hoặc `5` nếu `val_f1` vẫn tăng).
- **`MAX_LENGTH`**: `512` (Độ dài sequence tối đa mà các model này hỗ trợ).
- **`WEIGHT_DECAY`**: `0.01` (Giá trị tiêu chuẩn cho AdamW).
- **`WARMUP_RATIO`**: `0.1` (10% tổng số training step dùng để warmup learning rate).
- **`EFFECTIVE_BATCH_SIZE`**: `32` hoặc `64`.
  - Đây là batch size "hiệu quả" (`BATCH_SIZE` * `GRADIENT_ACCUMULATION_STEPS`).
  - *Ví dụ (VRAM 16-24GB):* Thử `BATCH_SIZE = 4`, `GRADIENT_ACCUMULATION_STEPS = 8` (-> 32).
  - *Ví dụ (VRAM 12GB):* Thử `BATCH_SIZE = 2`, `GRADIENT_ACCUMULATION_STEPS = 16` (-> 32).

## 2. Cấu hình Learning Rate (LR) (Rất quan trọng)

Các model khác nhau cần LR khác nhau.

| Nhóm Model | Tên Model | Lý do | Learning Rate (LR) |
| :--- | :--- | :--- | :--- |
| **Nhóm 1: Model Nền (Base)** | `uitnlp/CafeBERT`<br>`microsoft/infoxlm-large`<br>`FacebookAI/xlm-roberta-large` | Phải học tác vụ NLI từ đầu. | `2e-5` (Có thể thử `3e-5` cho CafeBERT) |
| **Nhóm 2: Đã tune NLI (Large)**| `joeddav/xlm-roberta-large-xnli`<br>`MoritzLaurer/ernie-m-large-mnli-xnli`<br>`MoritzLaurer/DeBERTa-v3-large-mnli...` | Chỉ "tiếp tục fine-tune" (continue) trên domain mới. Cần LR thấp hơn. | `1e-5` (Hoặc `8e-6`) |
| **Nhóm 3: Đã tune NLI (XLarge)**| `microsoft/deberta-xlarge-mnli` | Model cực lớn (1.5B), rất nhạy cảm với LR cao. **Bắt buộc** LR rất thấp. | `5e-6` |

---

## 3. Lưu ý CỰC KỲ QUAN TRỌNG

### A. Ánh xạ Nhãn (LABEL_MAP)

Đây là điều **quan trọng nhất** để thành công với các model Nhóm 2 và 3. Bạn phải ánh xạ nhãn của cuộc thi (`no`, `intrinsic`, `extrinsic`) sang các nhãn NLI gốc của model.

Mapping chuẩn (dùng cho DeBERTa, XLM-R, v.v.):
- `entailment` (hội thoại) $\rightarrow$ `ID = 0`
- `neutral` (trung lập) $\rightarrow$ `ID = 1`
- `contradiction` (mâu thuẫn) $\rightarrow$ `ID = 2`

Do đó, `LABEL_MAP` của bạn **bắt buộc** phải là:

```python
LABEL_MAP = {
    "no": 0,          # no (phù hợp) == entailment (0)
    "extrinsic": 1,   # extrinsic (bịa) == neutral (1)
    "intrinsic": 2    # intrinsic (mâu thuẫn) == contradiction (2)
}

# Cấu hình Hyperparameters (Phiên bản Cập nhật)

Dựa trên kết quả thử nghiệm thực tế, đây là các cấu hình đã được điều chỉnh.

## 1. Cấu hình Chung (Áp dụng cho tất cả model)

Các giá trị này đã được chứng minh là hoạt động tốt.

| Hyperparameter | Giá trị Đề xuất | Ghi chú |
| :--- | :--- | :--- |
| **`MAX_LENGTH`** | `512` | Độ dài sequence tối đa. |
| **`WEIGHT_DECAY`** | `0.01` | Giá trị tiêu chuẩn cho AdamW. |
| **`SCHEDULER_TYPE`** | `"linear"` | **Nên dùng.** Hội tụ tốt hơn "constant". |
| **`LABEL_SMOOTHING`** | `0.05` | Giúp chống overfitting. Đã dùng trong run thành công. |
| **`CLASS_WEIGHTS`** | `[1.039, ...]` | **Nên dùng.** Xử lý mất cân bằng data. |
| **`WARMUP_STEPS_OR_RATIO`** | `0.1` | 10% tổng số training step dùng để warmup. |
| **`RANDOM_STATE`** | `42` | Đảm bảo kết quả có thể tái lập. |

---

## 2. Cấu hình Riêng (Theo từng Nhóm Model)

Đây là phần được cập nhật quan trọng nhất.

### ⚠️ Trường hợp Đặc biệt: `microsoft/infoxlm-large` (Nhóm 1)

**Chiến lược:** Model này **rất nhạy cảm**. Nó yêu cầu LR và Batch Size rất nhỏ để hội tụ. Config này dựa trên lần train thành công (16-10) cho F1=0.7910.

| Hyperparameter | Giá trị Đề xuất (Đã kiểm chứng) | Ghi chú |
| :--- | :--- | :--- |
| **`LEARNING_RATE`** | `6e-06` | **Quan trọng nhất.** `2e-5` sẽ gây "Mode Collapse". |
| **`EPOCHS`** | `10` | Cần nhiều epoch (nhưng có Early Stopping). |
| **`BATCH_SIZE`** | `4` | |
| **`GRADIENT_ACCUMULATION_STEPS`** | `4` | $\rightarrow$ **Effective Batch Size = 16**. (Không dùng 32) |
| **`PATIENCE_LIMIT`** | `2` | **Rất quan trọng.** Đã cứu model khỏi overfitting. |
| **`CLASSIFIER_DROPOUT`** | `0.02` | Dùng dropout rất nhỏ (theo config 16-10). |
| **`LABEL_MAP`** | `{'intrinsic': 0, 'extrinsic': 1, 'no': 2}` | Dùng y hệt config đã thành công. |

---

### Nhóm 2: Model NLI-tuned (Large)

- **Models:** `joeddav/xlm-roberta-large-xnli`, `MoritzLaurer/ernie-m-large-mnli-xnli`, `MoritzLaurer/DeBERTa-v3-large-mnli...`
- **Chiến lược:** Các model này **đã biết NLI**. Chúng ta đang "fine-tune tiếp". Chúng hội tụ nhanh hơn và yêu cầu `LABEL_MAP` chuẩn NLI.

| Hyperparameter | Giá trị Đề xuất | Ghi chú |
| :--- | :--- | :--- |
| **`LABEL_MAP`** | `{"no": 0, "extrinsic": 1, "intrinsic": 2}` | **BẮT BUỘC** dùng map chuẩn NLI. |
| **`LEARNING_RATE`** | `1e-5` | LR thấp vì model đã "khôn". |
| **`EPOCHS`** | `3` (hoặc `4`) | Hội tụ rất nhanh, không cần nhiều epoch. |
| **`EFFECTIVE_BATCH_SIZE`** | `32` | (`BATCH_SIZE=4`, `GRAD_ACCUM=8`) |
| **`PATIENCE_LIMIT`** | `3` | |

---

### Nhóm 3: Model NLI-tuned (XLarge)

- **Model:** `microsoft/deberta-xlarge-mnli`
- **Chiến lược:** Tương tự Nhóm 2 nhưng cần cẩn thận hơn nữa vì model quá lớn.

| Hyperparameter | Giá trị Đề xuất | Ghi chú |
| :--- | :--- | :--- |
| **`LABEL_MAP`** | `{"no": 0, "extrinsic": 1, "intrinsic": 2}` | **BẮT BUỘC** dùng map chuẩn NLI. |
| **`LEARNING_RATE`** | `5e-6` | **Siêu nhỏ**. LR cao hơn sẽ hỏng. |
| **`EPOCHS`** | `3` | |
| **`BATCH_SIZE`** | `1` | **Bắt buộc** (trừ khi có GPU A100 40GB+). |
| **`GRADIENT_ACCUMULATION_STEPS`** | `32` | Để giữ `Effective Batch Size = 32`. |
| **`PATIENCE_LIMIT`** | `3` | |

In [ ]:
# ==============================================================================
# CELL 2: CODE - CONFIGURATION
# ==============================================================================
import os

MODEL_NAMES = [
    "joeddav/xlm-roberta-large-xnli",
    "microsoft/infoxlm-large",
    "uitnlp/CafeBERT",
    "FacebookAI/xlm-roberta-large",
    "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli",
    "MoritzLaurer/ernie-m-large-mnli-xnli",
    "microsoft/deberta-xlarge-mnli",
]


class Config:
    # --- PATHS ---
    ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), "../../"))
    DATA_DIR = os.path.join(ROOT_DIR, "data")
    TRAIN_FILE = os.path.join(DATA_DIR, "vihallu-train.csv")
    TEST_FILE = os.path.join(DATA_DIR, "vihallu-public-test.csv")
    SUBMISSION_DIR = os.path.join(ROOT_DIR, "submission")
    SUBMISSION_CSV = "submit.csv"
    SUBMISSION_ZIP = "submit.zip"

    # --- MODEL SELECTION ---
    MODEL_NAME = MODEL_NAMES[2]  # Thay đổi index để chọn model khác
    MODEL_OUTPUT_DIR_BASE = os.path.join(
        ROOT_DIR, "models", f"{MODEL_NAME.split('/')[-1]}-kfold"
    )

    # --- K-FOLD CONFIGURATION ---
    K_FOLDS = 5  # Số fold bạn muốn chạy (phổ biến là 5 hoặc 10)

    # --- TRAINING HYPERPARAMETERS ---
    MAX_LENGTH = 512
    RANDOM_STATE = 42
    EPOCHS = 8
    BATCH_SIZE = 4
    GRADIENT_ACCUMULATION_STEPS = 8
    SCHEDULER_TYPE = "cosine"
    LEARNING_RATE = 2e-5
    WEIGHT_DECAY = 0.03
    LABEL_SMOOTHING = 0.05
    WARMUP_STEPS_OR_RATIO = 0.1
    EPSILON = 1e-8
    PATIENCE_LIMIT = 2  # Early stopping patience

    # --- LABEL MAPPING & CLASS WEIGHTS ---
    LABEL_MAP = {"no": 0, "extrinsic": 1, "intrinsic": 2}
    ID2LABEL = {v: k for k, v in LABEL_MAP.items()}
    CLASS_WEIGHTS = [1.0393466963622866, 1.0114145354717525, 0.9531590413943355]


cfg = Config()


# LOGGER

In [ ]:
import logging
import os
from datetime import datetime

# Thư mục gốc để lưu tất cả các file log
LOG_BASE_DIR = "logs"

# Dùng một dictionary để lưu các logger đã tạo, tránh việc tạo lại và gây ra log trùng lặp
_loggers = {}


def setup_logger(model_name: str, log_level=logging.INFO):
    """
    Thiết lập và trả về một logger để ghi log vào cả console và file.

    - Mỗi model sẽ có một thư mục log riêng dựa trên `model_name`.
    - Mỗi lần chạy sẽ tạo một file log mới có tên là timestamp (ví dụ: 2023-10-27_15-30-00.log).
    - Đảm bảo không có log nào bị ghi đè.

    Args:
        model_name (str): Tên của model, dùng để tạo thư mục con. Ví dụ: 'xnli-large-tuned'.
        log_level (int): Cấp độ log, mặc định là logging.INFO.

    Returns:
        logging.Logger: Instance của logger đã được cấu hình.
    """
    # Nếu logger cho model này đã tồn tại, trả về nó ngay lập tức
    if model_name in _loggers:
        return _loggers[model_name]

    # Xử lý tên model để an toàn khi tạo tên thư mục (thay thế "/")
    safe_model_name = model_name.replace("/", "_").replace("\\", "_")
    model_log_dir = os.path.join(LOG_BASE_DIR, safe_model_name)
    os.makedirs(model_log_dir, exist_ok=True)

    # Tạo logger
    logger = logging.getLogger(safe_model_name)
    logger.setLevel(log_level)

    # Ngăn không cho log lan truyền đến root logger để tránh in ra console 2 lần
    logger.propagate = False

    # Định dạng cho log message
    formatter = logging.Formatter(
        "%(asctime)s - [%(levelname)s] - %(message)s", datefmt="%Y-%m-%d %H:%M:%S"
    )

    # Tạo File Handler để ghi log ra file
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    log_file_path = os.path.join(model_log_dir, f"{timestamp}.log")

    file_handler = logging.FileHandler(log_file_path, encoding="utf-8")
    file_handler.setLevel(log_level)
    file_handler.setFormatter(formatter)

    # Tạo Console (Stream) Handler để in log ra màn hình
    console_handler = logging.StreamHandler()
    console_handler.setLevel(log_level)
    console_handler.setFormatter(formatter)

    # Thêm các handler vào logger
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

    # Lưu logger vào cache
    _loggers[model_name] = logger

    logger.info(
        f"Logger cho '{safe_model_name}' đã được khởi tạo. File log: {log_file_path}"
    )

    return logger


## Setup logger

In [ ]:
logger = setup_logger(f"{cfg.MODEL_NAME}-training-kfold")
logger.info(f"Logger initialized for {cfg.MODEL_NAME}")

logger.info("=" * 60)
logger.info("🚀 STARTING TRAINING SESSION")
logger.info("=" * 60)
for key, value in Config.__dict__.items():
    if not key.startswith("__") and not callable(value):
        logger.info(f"{key}: {value}")
logger.info("=" * 60)


2025-10-18 14:59:20 - [INFO] - Logger cho 'uitnlp_CafeBERT-training-kfold' đã được khởi tạo. File log: logs/uitnlp_CafeBERT-training-kfold/2025-10-18_14-59-20.log
2025-10-18 14:59:20 - [INFO] - Logger initialized for uitnlp/CafeBERT
2025-10-18 14:59:20 - [INFO] - ============================================================
2025-10-18 14:59:20 - [INFO] - 🚀 STARTING TRAINING SESSION
2025-10-18 14:59:20 - [INFO] - ============================================================
2025-10-18 14:59:20 - [INFO] - ROOT_DIR: /home/guest/Projects/CS221
2025-10-18 14:59:20 - [INFO] - DATA_DIR: /home/guest/Projects/CS221/data
2025-10-18 14:59:20 - [INFO] - TRAIN_FILE: /home/guest/Projects/CS221/data/vihallu-train.csv
2025-10-18 14:59:20 - [INFO] - TEST_FILE: /home/guest/Projects/CS221/data/vihallu-public-test.csv
2025-10-18 14:59:20 - [INFO] - SUBMISSION_DIR: /home/guest/Projects/CS221/submission
2025-10-18 14:59:20 - [INFO] - SUBMISSION_CSV: submit.csv
2025-10-18 14:59:20 - [INFO] - SUBMISSION_ZIP: su

# UTILS & CORE CLASSES

Các lớp và hàm tiện ích cốt lõi như Dataset, Model Loading, Training/Evaluation functions.

In [4]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from dotenv import load_dotenv
from huggingface_hub import login
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from tqdm.contrib.logging import logging_redirect_tqdm
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_scheduler,
)
from torch.optim import AdamW


# Hallucination Dataset

In [5]:
class HallucinationDataset(Dataset):
    def __init__(self, premises, hypotheses, labels, tokenizer, max_len):
        self.premises = premises
        self.hypotheses = hypotheses
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoding = self.tokenizer.encode_plus(
            self.premises[idx],
            self.hypotheses[idx],
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            truncation=True,
            return_attention_mask=True,
        )
        return {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
            "labels": self.labels[idx],
        }


class HallucinationTestDataset(Dataset):
    def __init__(self, premises, hypotheses, tokenizer, max_len):
        self.premises = premises
        self.hypotheses = hypotheses
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.premises)

    def __getitem__(self, idx):
        encoding = self.tokenizer.encode_plus(
            self.premises[idx],
            self.hypotheses[idx],
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            truncation=True,
            return_attention_mask=True,
        )
        return {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
        }


## DATA PREPARATION FUNCTIONS

In [6]:
def prepare_full_data(config, logger):
    df = pd.read_csv(config.TRAIN_FILE)
    logger.info(f"✅ Đọc thành công {len(df)} mẫu từ file: {config.TRAIN_FILE}")
    df["premise"] = (
        "Câu hỏi: "
        + df["prompt"].astype(str)
        + " Ngữ cảnh: "
        + df["context"].astype(str)
    )
    df["hypothesis"] = df["response"].astype(str)
    df["label_id"] = df["label"].map(config.LABEL_MAP)
    df.dropna(subset=["label_id"], inplace=True)
    df["label_id"] = df["label_id"].astype(int)
    return df


def prepare_test_data(config, logger):
    try:
        df = pd.read_csv(config.TEST_FILE)
        logger.info(f"✅ Đọc thành công {len(df)} mẫu từ file test: {config.TEST_FILE}")
    except FileNotFoundError:
        logger.error(f"File test không tồn tại: {config.TEST_FILE}")
        return None, None
    ids = df["id"].tolist()
    df["premise"] = (
        "Câu hỏi: "
        + df["prompt"].astype(str)
        + " Ngữ cảnh: "
        + df["context"].astype(str)
    )
    df["hypothesis"] = df["response"].astype(str)
    return df, ids


## MODEL LOADING

In [7]:
def get_model_and_tokenizer(config):
    logger.info(f"Đang tải model: {config.MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME, trust_remote_code=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        config.MODEL_NAME, num_labels=len(config.LABEL_MAP), trust_remote_code=True
    )
    return model, tokenizer


# TRAINING & EVALUATION FUNCTIONS

In [8]:
def train_one_epoch(
    model,
    data_loader,
    loss_fn,
    optimizer,
    scheduler,
    device,
    gradient_accumulation_steps,
):
    model.train()
    total_loss = 0
    progress_bar = tqdm(data_loader, desc="Training", leave=False, dynamic_ncols=True)
    optimizer.zero_grad()
    for step, batch in enumerate(progress_bar):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        total_loss += loss.item()
        scaled_loss = loss / gradient_accumulation_steps
        scaled_loss.backward()

        if (step + 1) % gradient_accumulation_steps == 0 or (step + 1) == len(
            data_loader
        ):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
    return total_loss / len(data_loader)


In [9]:
def evaluate(model, data_loader, loss_fn, device):
    model.eval()
    all_preds, all_labels = [], []
    total_val_loss = 0
    progress_bar = tqdm(data_loader, desc="Evaluating", leave=False, dynamic_ncols=True)
    with torch.no_grad():
        for batch in progress_bar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            logits = outputs.logits

            total_val_loss += loss.item()
            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = total_val_loss / len(data_loader)
    return all_labels, all_preds, avg_val_loss


## HÀM HUẤN LUYỆN CHO MỘT FOLD

In [10]:
def train_one_fold(fold_num, train_df, val_df, cfg, logger):
    """
    Hàm này chứa toàn bộ logic để huấn luyện và đánh giá trên một fold duy nhất.
    Nó sẽ tải lại model, tạo dataloader, huấn luyện, và lưu lại model tốt nhất cho fold này.

    Args:
        fold_num (int): Số thứ tự của fold (ví dụ: 1, 2, ...).
        train_df (pd.DataFrame): DataFrame chứa dữ liệu huấn luyện cho fold này.
        val_df (pd.DataFrame): DataFrame chứa dữ liệu đánh giá cho fold này.
        cfg (Config): Đối tượng chứa toàn bộ cấu hình.
        logger (Logger): Đối tượng logger để ghi log.

    Returns:
        float: Điểm Macro-F1 tốt nhất đạt được trên tập validation của fold này.
    """
    # 1. Tải lại model và tokenizer để đảm bảo mỗi fold là độc lập
    model, tokenizer = get_model_and_tokenizer(cfg)

    # 2. Tạo Dataset và DataLoader
    train_dataset = HallucinationDataset(
        train_df["premise"].to_list(),
        train_df["hypothesis"].to_list(),
        train_df["label_id"].to_list(),
        tokenizer,
        cfg.MAX_LENGTH,
    )
    val_dataset = HallucinationDataset(
        val_df["premise"].to_list(),
        val_df["hypothesis"].to_list(),
        val_df["label_id"].to_list(),
        tokenizer,
        cfg.MAX_LENGTH,
    )
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    train_loader = DataLoader(
        train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True, collate_fn=data_collator
    )
    val_loader = DataLoader(
        val_dataset, batch_size=cfg.BATCH_SIZE, collate_fn=data_collator
    )

    # 3. Thiết lập Optimizer, Scheduler, Loss
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = AdamW(
        model.parameters(),
        lr=cfg.LEARNING_RATE,
        weight_decay=cfg.WEIGHT_DECAY,
        eps=cfg.EPSILON,
    )

    num_training_steps = (
        len(train_loader) // cfg.GRADIENT_ACCUMULATION_STEPS * cfg.EPOCHS
    )
    warmup_steps = int(cfg.WARMUP_STEPS_OR_RATIO * num_training_steps)
    scheduler = get_scheduler(
        cfg.SCHEDULER_TYPE,
        optimizer=optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=num_training_steps,
    )

    class_weights = torch.tensor(cfg.CLASS_WEIGHTS, dtype=torch.float).to(device)
    loss_fn = nn.CrossEntropyLoss(
        weight=class_weights, label_smoothing=cfg.LABEL_SMOOTHING
    ).to(device)

    # 4. Vòng lặp huấn luyện cho fold
    best_macro_f1 = 0.0
    patience_counter = 0
    for epoch in range(cfg.EPOCHS):
        epoch_num = epoch + 1
        logger.info(f"--- [Fold {fold_num}] Epoch {epoch_num}/{cfg.EPOCHS} ---")

        # Huấn luyện
        avg_train_loss = train_one_epoch(
            model,
            train_loader,
            loss_fn,
            optimizer,
            scheduler,
            device,
            cfg.GRADIENT_ACCUMULATION_STEPS,
        )

        # Đánh giá
        val_labels, val_preds, avg_val_loss = evaluate(
            model, val_loader, loss_fn, device
        )
        macro_f1 = f1_score(val_labels, val_preds, average="macro")

        logger.info(
            f"[Fold {fold_num} | Epoch {epoch_num}] Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val Macro-F1: {macro_f1:.4f}"
        )

        # In Classification Report
        target_names = [cfg.ID2LABEL[i] for i in range(len(cfg.LABEL_MAP))]
        report = classification_report(
            val_labels, val_preds, target_names=target_names, digits=4
        )
        logger.info(f"Classification Report (Epoch {epoch_num}):\n{report}")

        # Lưu model tốt nhất và kiểm tra early stopping
        if macro_f1 > best_macro_f1:
            best_macro_f1 = macro_f1
            patience_counter = 0
            model_output_dir_fold = os.path.join(
                cfg.MODEL_OUTPUT_DIR_BASE, f"fold_{fold_num}"
            )
            os.makedirs(model_output_dir_fold, exist_ok=True)
            model.save_pretrained(model_output_dir_fold)
            tokenizer.save_pretrained(model_output_dir_fold)
            logger.info(
                f"🎉 Best F1 improved to {best_macro_f1:.4f}. Saving model for Fold {fold_num}!"
            )
        else:
            patience_counter += 1
            if patience_counter >= cfg.PATIENCE_LIMIT:
                logger.warning(
                    f"Early stopping at Epoch {epoch_num} for Fold {fold_num}."
                )
                break

    logger.info(f"🏁 Finished Fold {fold_num}. Best Macro-F1: {best_macro_f1:.4f}")
    return best_macro_f1


# PART 1: K-FOLD TRAINING

In [11]:
# Tải biến môi trường từ file envs/.env.
dotenv_path = os.path.join(os.getcwd(), "envs", ".env")
load_dotenv(dotenv_path)
print(f"dotenv_path: {dotenv_path}")


dotenv_path: /home/guest/Projects/CS221/envs/.env


In [12]:
# lấy HF token để login
hf_token = os.getenv("HUGGING_FACE_TOKEN")

if hf_token:
    print("INFO: Tìm thấy HUGGING_FACE_TOKEN. Đang đăng nhập...")
    login(token=hf_token)
    print("INFO: Đăng nhập Hugging Face thành công.")
else:
    print(
        "WARNING: Không tìm thấy HUGGING_FACE_TOKEN trong file .env. Một số model có thể yêu cầu đăng nhập."
    )


INFO: Tìm thấy HUGGING_FACE_TOKEN. Đang đăng nhập...
INFO: Đăng nhập Hugging Face thành công.


In [13]:
# --- Start Training ---
logger.info("=" * 60)
logger.info("🚀 STARTING K-FOLD TRAINING SESSION")
logger.info("=" * 60)


2025-10-18 14:59:23 - [INFO] - ============================================================
2025-10-18 14:59:23 - [INFO] - 🚀 STARTING K-FOLD TRAINING SESSION
2025-10-18 14:59:23 - [INFO] - ============================================================
2025-10-18 14:59:23 - [INFO] - 🚀 STARTING K-FOLD TRAINING SESSION
2025-10-18 14:59:23 - [INFO] - ============================================================


# 1. Chuẩn bị dữ liệu

In [14]:
full_df = prepare_full_data(cfg, logger)
skf = StratifiedKFold(n_splits=cfg.K_FOLDS, shuffle=True, random_state=cfg.RANDOM_STATE)
all_fold_f1_scores = []


2025-10-18 14:59:23 - [INFO] - ✅ Đọc thành công 7000 mẫu từ file: /home/guest/Projects/CS221/data/vihallu-train.csv


# 2. Bắt đầu vòng lặp K-Fold

In [15]:
for fold, (train_idx, val_idx) in enumerate(skf.split(full_df, full_df["label_id"])):
    fold_num = fold + 1
    logger.info(f"\n{'='*25} STARTING FOLD {fold_num}/{cfg.K_FOLDS} {'='*25}")

    # Chia dữ liệu cho fold hiện tại
    train_df = full_df.iloc[train_idx]
    val_df = full_df.iloc[val_idx]

    # Gọi hàm để huấn luyện cho fold này
    best_f1_for_fold = train_one_fold(fold_num, train_df, val_df, cfg, logger)

    # Lưu lại kết quả
    all_fold_f1_scores.append(best_f1_for_fold)


2025-10-18 14:59:23 - [INFO] - 
========================= STARTING FOLD 1/5 =========================
2025-10-18 14:59:23 - [INFO] - Đang tải model: uitnlp/CafeBERT
Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at uitnlp/CafeBERT and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
2025-10-18 14:59:25 - [INFO] - --- [Fold 1] Epoch 1/8 ---


Training:   0%|          | 0/1400 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/350 [00:00<?, ?it/s]

2025-10-18 15:03:58 - [INFO] - [Fold 1 | Epoch 1] Train Loss: 1.0197, Val Loss: 0.8147, Val Macro-F1: 0.6603
2025-10-18 15:03:58 - [INFO] - Classification Report (Epoch 1):
              precision    recall  f1-score   support

          no     0.8393    0.6281    0.7185       449
   extrinsic     0.6202    0.8658    0.7227       462
   intrinsic     0.5847    0.5010    0.5396       489

    accuracy                         0.6621      1400
   macro avg     0.6814    0.6650    0.6603      1400
weighted avg     0.6781    0.6621    0.6574      1400

2025-10-18 15:04:00 - [INFO] - 🎉 Best F1 improved to 0.6603. Saving model for Fold 1!
2025-10-18 15:04:00 - [INFO] - --- [Fold 1] Epoch 2/8 ---


Training:   0%|          | 0/1400 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/350 [00:00<?, ?it/s]

2025-10-18 16:54:37 - [INFO] - [Fold 5 | Epoch 4] Train Loss: 0.3559, Val Loss: 0.7519, Val Macro-F1: 0.7676
2025-10-18 16:54:37 - [INFO] - Classification Report (Epoch 4):
              precision    recall  f1-score   support

          no     0.7866    0.8864    0.8335       449
   extrinsic     0.7329    0.7679    0.7500       461
   intrinsic     0.7883    0.6612    0.7192       490

    accuracy                         0.7686      1400
   macro avg     0.7693    0.7718    0.7676      1400
weighted avg     0.7695    0.7686    0.7660      1400

2025-10-18 16:54:37 - [INFO] - --- [Fold 5] Epoch 5/8 ---


Training:   0%|          | 0/1400 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/350 [00:00<?, ?it/s]

2025-10-18 16:59:01 - [INFO] - [Fold 5 | Epoch 5] Train Loss: 0.2205, Val Loss: 0.8668, Val Macro-F1: 0.7721
2025-10-18 16:59:01 - [INFO] - Classification Report (Epoch 5):
              precision    recall  f1-score   support

          no     0.8386    0.8330    0.8358       449
   extrinsic     0.7705    0.7137    0.7410       461
   intrinsic     0.7135    0.7673    0.7394       490

    accuracy                         0.7707      1400
   macro avg     0.7742    0.7713    0.7721      1400
weighted avg     0.7724    0.7707    0.7708      1400

2025-10-18 16:59:01 - [WARNING] - Early stopping at Epoch 5 for Fold 5.
2025-10-18 16:59:01 - [INFO] - 🏁 Finished Fold 5. Best Macro-F1: 0.7799
2025-10-18 16:59:01 - [INFO] - Classification Report (Epoch 5):
              precision    recall  f1-score   support

          no     0.8386    0.8330    0.8358       449
   extrinsic     0.7705    0.7137    0.7410       461
   intrinsic     0.7135    0.7673    0.7394       490

    accuracy        

# 3. Tổng kết

In [16]:
logger.info(f"{'='*25} K-FOLD TRAINING SUMMARY {'='*25}")
logger.info(f"F1 Scores per fold: {all_fold_f1_scores}")
logger.info(
    f"📈 Average F1-Score: {np.mean(all_fold_f1_scores):.4f} ± {np.std(all_fold_f1_scores):.4f}"
)


2025-10-18 16:59:01 - [INFO] - ========================= K-FOLD TRAINING SUMMARY =========================
2025-10-18 16:59:01 - [INFO] - F1 Scores per fold: [0.7846514340882079, 0.7898002761211868, 0.7812753201996993, 0.773504476741135, 0.7798532612613469]
2025-10-18 16:59:01 - [INFO] - 📈 Average F1-Score: 0.7818 ± 0.0054
2025-10-18 16:59:01 - [INFO] - F1 Scores per fold: [0.7846514340882079, 0.7898002761211868, 0.7812753201996993, 0.773504476741135, 0.7798532612613469]
2025-10-18 16:59:01 - [INFO] - 📈 Average F1-Score: 0.7818 ± 0.0054


# PART 2: INFERENCE & ENSEMBLING

## PREDICTION FUNCTION

In [17]:
# def predict_probabilities(model, data_loader, device):
#     model.eval()
#     all_probs = []
#     progress_bar = tqdm(data_loader, desc="Predicting", leave=False, dynamic_ncols=True)
#     with torch.no_grad():
#         for batch in progress_bar:
#             input_ids = torch.tensor(batch["input_ids"]).to(device)
#             attention_mask = torch.tensor(batch["attention_mask"]).to(device)
#             outputs = model(input_ids=input_ids, attention_mask=attention_mask)
#             probs = F.softmax(outputs.logits, dim=-1)
#             all_probs.append(probs.cpu().numpy())
#     return np.concatenate(all_probs, axis=0)


## MAIN INFERENCE SCRIPT

In [18]:
# ogger.info(f"\n{'='*25} INFERENCE & ENSEMBLING {'='*25}")

# # 1. Chuẩn bị dữ liệu test
# test_df, test_ids = prepare_test_data(cfg, logger)

# if test_df is not None:
#     _, tokenizer = get_model_and_tokenizer(cfg)  # Tải tokenizer
#     test_dataset = HallucinationTestDataset(
#         test_df["premise"].to_list(),
#         test_df["hypothesis"].to_list(),
#         tokenizer,
#         cfg.MAX_LENGTH,
#     )
#     data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
#     test_loader = DataLoader(
#         test_dataset,
#         batch_size=cfg.BATCH_SIZE * 2,
#         shuffle=False,
#         collate_fn=data_collator,
#     )

#     # 2. Thu thập dự đoán từ mỗi fold
#     all_fold_probabilities = []
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#     for fold in range(cfg.K_FOLDS):
#         fold_num = fold + 1
#         model_path = os.path.join(cfg.MODEL_OUTPUT_DIR_BASE, f"fold_{fold_num}")
#         if not os.path.exists(model_path):
#             logger.warning(f"Model for fold {fold_num} not found. Skipping.")
#             continue

#         logger.info(f"--- Loading model from Fold {fold_num} for prediction ---")
#         model = AutoModelForSequenceClassification.from_pretrained(model_path)
#         model.to(device)

#         fold_probs = predict_probabilities(model, test_loader, device)
#         all_fold_probabilities.append(fold_probs)

#         del model
#         torch.cuda.empty_cache()

#     # 3. Ensembling và tạo submission
#     if all_fold_probabilities:
#         logger.info("Averaging probabilities (Soft Voting)...")
#         avg_probabilities = np.mean(all_fold_probabilities, axis=0)
#         final_predictions_ids = np.argmax(avg_probabilities, axis=1)
#         final_predictions_labels = [
#             cfg.ID2LABEL[pred_id] for pred_id in final_predictions_ids
#         ]

#         submission_df = pd.DataFrame(
#             {"id": test_ids, "label": final_predictions_labels}
#         )

#         os.makedirs(cfg.SUBMISSION_DIR, exist_ok=True)
#         csv_path = os.path.join(cfg.SUBMISSION_DIR, cfg.SUBMISSION_CSV)
#         submission_df.to_csv(csv_path, index=False)
#         logger.info(f"✅ Submission CSV saved to: {csv_path}")

#         zip_path = os.path.join(cfg.SUBMISSION_DIR, cfg.SUBMISSION_ZIP)
#         with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
#             zipf.write(csv_path, os.path.basename(csv_path))
#         logger.info(f"✅ Submission ZIP created at: {zip_path}")
#     else:
#         logger.error("No models found. Submission not created.")

# logger.info("🏁 PROCESS COMPLETE 🏁")
